# Covered Call (Buy-Write) Options Strategy

## Overview

A **covered call** (also known as buy-write) is an options strategy where you:
1. **Own 100 shares** of the underlying stock
2. **Sell (write) 1 call option** on those shares

This strategy generates income from the option premium while potentially limiting upside gains.

### Key Characteristics
- **Direction**: Neutral to slightly bullish
- **Risk**: Limited downside (can lose stock value minus premium)
- **Reward**: Limited upside (capped at strike price plus premium)
- **Best Used**: When you expect the stock to remain flat or increase slightly

## Learning Objectives
In this tutorial, we will:
1. Understand the mechanics of covered calls
2. Calculate profit/loss scenarios
3. Visualize payoff diagrams
4. Compare different strike prices
5. Analyze real-world examples

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Part 1: Strategy Mechanics

### Example Setup
Let's use a practical example:
- **Stock**: XYZ Corp
- **Current Stock Price**: $50
- **Position**: Own 100 shares
- **Action**: Sell 1 call option with strike price $55
- **Premium Received**: $2 per share ($200 total)
- **Expiration**: 30 days

In [ ]:
# Define our covered call parameters
stock_purchase_price = 50
shares = 100
strike_price = 55
premium_per_share = 2
total_premium = premium_per_share * shares

print(f"=" * 50)
print(f"COVERED CALL POSITION")
print(f"=" * 50)
print(f"Stock Purchase Price: ${stock_purchase_price}")
print(f"Shares Owned: {shares}")
print(f"Call Strike Price: ${strike_price}")
print(f"Premium per Share: ${premium_per_share}")
print(f"Total Premium Received: ${total_premium}")
print(f"=" * 50)

## Part 2: Profit/Loss Calculations

### Understanding the Outcomes

At expiration, three scenarios are possible:

1. **Stock price < Strike price (Out of the Money)**
   - Option expires worthless
   - You keep the stock and the premium
   - Profit/Loss = (Stock Price - Purchase Price) × 100 + Premium

2. **Stock price = Strike price (At the Money)**
   - Option expires worthless or barely ITM
   - You keep the stock and premium

3. **Stock price > Strike price (In the Money)**
   - Option will be exercised
   - Stock sold at strike price
   - Profit/Loss = (Strike - Purchase Price) × 100 + Premium

In [ ]:
def calculate_covered_call_pnl(stock_price_at_expiry, stock_purchase_price, strike_price, premium_per_share, shares=100):
    """
    Calculate profit/loss for a covered call position at expiration.
    
    Parameters:
    -----------
    stock_price_at_expiry : float
        Stock price at option expiration
    stock_purchase_price : float
        Initial stock purchase price
    strike_price : float
        Call option strike price
    premium_per_share : float
        Premium received per share
    shares : int
        Number of shares (default 100)
    
    Returns:
    --------
    float : Total profit/loss for the position
    """
    # Premium received (income)
    premium_income = premium_per_share * shares
    
    # Stock profit/loss
    if stock_price_at_expiry > strike_price:
        # Option exercised - stock sold at strike price
        stock_pnl = (strike_price - stock_purchase_price) * shares
    else:
        # Option expires worthless - keep stock
        stock_pnl = (stock_price_at_expiry - stock_purchase_price) * shares
    
    total_pnl = stock_pnl + premium_income
    return total_pnl

In [ ]:
# Test different scenarios
scenarios = [
    {"name": "Significant Drop", "price": 40},
    {"name": "Moderate Drop", "price": 45},
    {"name": "Small Drop (Breakeven)", "price": 48},
    {"name": "No Change", "price": 50},
    {"name": "Moderate Gain", "price": 53},
    {"name": "At Strike", "price": 55},
    {"name": "Above Strike", "price": 60},
    {"name": "Well Above Strike", "price": 65}
]

print(f"\n{'Scenario':<25} {'Stock Price':<15} {'P/L':<15} {'Return %':<15}")
print("="*70)

for scenario in scenarios:
    price = scenario['price']
    pnl = calculate_covered_call_pnl(price, stock_purchase_price, strike_price, premium_per_share)
    initial_investment = stock_purchase_price * shares
    return_pct = (pnl / initial_investment) * 100
    
    print(f"{scenario['name']:<25} ${price:<14.2f} ${pnl:<14.2f} {return_pct:>6.2f}%")

## Part 3: Payoff Diagram Visualization

The payoff diagram helps visualize the profit/loss across all possible stock prices at expiration.

In [ ]:
# Create a range of possible stock prices
stock_prices = np.linspace(30, 70, 200)

# Calculate P/L for each price point
pnl_values = [calculate_covered_call_pnl(price, stock_purchase_price, strike_price, premium_per_share) 
              for price in stock_prices]

# Create the payoff diagram
fig, ax = plt.subplots(figsize=(12, 7))

# Plot the P/L line
ax.plot(stock_prices, pnl_values, linewidth=2.5, color='#2E86AB', label='Covered Call P/L')

# Add horizontal line at zero (breakeven)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Breakeven')

# Add vertical lines for key prices
ax.axvline(x=stock_purchase_price, color='green', linestyle='--', linewidth=1, alpha=0.5, label=f'Purchase Price (${stock_purchase_price})')
ax.axvline(x=strike_price, color='red', linestyle='--', linewidth=1, alpha=0.5, label=f'Strike Price (${strike_price})')

# Fill areas
ax.fill_between(stock_prices, pnl_values, 0, where=(np.array(pnl_values) > 0), 
                alpha=0.3, color='green', label='Profit Zone')
ax.fill_between(stock_prices, pnl_values, 0, where=(np.array(pnl_values) < 0), 
                alpha=0.3, color='red', label='Loss Zone')

# Labels and formatting
ax.set_xlabel('Stock Price at Expiration ($)', fontsize=12, fontweight='bold')
ax.set_ylabel('Profit / Loss ($)', fontsize=12, fontweight='bold')
ax.set_title('Covered Call Payoff Diagram', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

# Add annotations for max profit and max loss
max_profit = (strike_price - stock_purchase_price) * shares + total_premium
ax.annotate(f'Max Profit: ${max_profit:.0f}', 
            xy=(strike_price + 5, max_profit), 
            fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgreen', alpha=0.7))

plt.tight_layout()
plt.show()

print(f"\nKey Metrics:")
print(f"Maximum Profit: ${max_profit:.2f}")
print(f"Breakeven Price: ${stock_purchase_price - premium_per_share:.2f}")
print(f"Downside Protection: ${total_premium:.2f} ({(premium_per_share/stock_purchase_price)*100:.1f}% of stock price)")

## Part 4: Comparing Different Strike Prices

Let's compare how different strike prices affect the covered call strategy. Generally:
- **Lower strike prices** = Higher premium but lower max profit
- **Higher strike prices** = Lower premium but higher max profit

In [ ]:
# Define different strike scenarios
strike_scenarios = [
    {"strike": 52, "premium": 3.00, "label": "ITM: Strike $52"},  # In-the-money
    {"strike": 50, "premium": 2.50, "label": "ATM: Strike $50"},  # At-the-money
    {"strike": 55, "premium": 2.00, "label": "OTM: Strike $55"},  # Out-of-the-money
    {"strike": 60, "premium": 0.75, "label": "Far OTM: Strike $60"}  # Far out-of-the-money
]

fig, ax = plt.subplots(figsize=(14, 8))
stock_prices = np.linspace(35, 70, 200)

colors = ['#E63946', '#F77F00', '#06AED5', '#2B9348']

for i, scenario in enumerate(strike_scenarios):
    pnl_values = [calculate_covered_call_pnl(price, stock_purchase_price, 
                                              scenario['strike'], scenario['premium']) 
                  for price in stock_prices]
    
    ax.plot(stock_prices, pnl_values, linewidth=2.5, 
            color=colors[i], label=scenario['label'], alpha=0.8)

# Add reference lines
ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(x=stock_purchase_price, color='gray', linestyle=':', linewidth=1.5, 
           alpha=0.5, label=f'Current Price (${stock_purchase_price})')

ax.set_xlabel('Stock Price at Expiration ($)', fontsize=12, fontweight='bold')
ax.set_ylabel('Profit / Loss ($)', fontsize=12, fontweight='bold')
ax.set_title('Covered Call Strategy: Comparing Different Strike Prices', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create comparison table
comparison_data = []

for scenario in strike_scenarios:
    strike = scenario['strike']
    premium = scenario['premium']
    
    max_profit = (strike - stock_purchase_price) * shares + premium * shares
    max_profit_pct = (max_profit / (stock_purchase_price * shares)) * 100
    breakeven = stock_purchase_price - premium
    downside_protection_pct = (premium / stock_purchase_price) * 100
    
    comparison_data.append({
        'Strike': f"${strike}",
        'Premium': f"${premium}",
        'Max Profit': f"${max_profit:.0f}",
        'Max Return %': f"{max_profit_pct:.1f}%",
        'Breakeven': f"${breakeven:.2f}",
        'Downside Protection': f"{downside_protection_pct:.1f}%"
    })

df_comparison = pd.DataFrame(comparison_data)
print("\nStrike Price Comparison:")
print(df_comparison.to_string(index=False))

## Part 5: Real-World Application

Let's simulate a more realistic scenario with monthly covered call writing over a 6-month period.

In [ ]:
def simulate_covered_call_strategy(initial_price, months=6, annual_growth=0.08, volatility=0.15):
    """
    Simulate a covered call strategy over multiple months.
    
    Parameters:
    -----------
    initial_price : float
        Starting stock price
    months : int
        Number of months to simulate
    annual_growth : float
        Expected annual stock growth rate
    volatility : float
        Annual volatility (std deviation)
    """
    np.random.seed(42)  # For reproducibility
    
    monthly_growth = annual_growth / 12
    monthly_vol = volatility / np.sqrt(12)
    
    results = []
    current_price = initial_price
    total_premium = 0
    shares_owned = 100
    
    for month in range(1, months + 1):
        # Simulate stock price movement (geometric Brownian motion)
        return_rate = np.random.normal(monthly_growth, monthly_vol)
        new_price = current_price * (1 + return_rate)
        
        # Set strike price slightly OTM (5% above current price)
        strike = current_price * 1.05
        
        # Estimate premium (simplified: ~2% of stock price for OTM call)
        premium_per_share = current_price * 0.02
        premium = premium_per_share * shares_owned
        total_premium += premium
        
        # Check if option was exercised
        exercised = new_price > strike
        
        results.append({
            'Month': month,
            'Start Price': current_price,
            'End Price': new_price,
            'Strike': strike,
            'Premium': premium,
            'Exercised': 'Yes' if exercised else 'No'
        })
        
        if exercised:
            # Stock was called away, but we can buy it back for next month
            current_price = new_price
        else:
            current_price = new_price
    
    df = pd.DataFrame(results)
    
    # Calculate total returns
    stock_appreciation = (current_price - initial_price) * shares_owned
    total_return = stock_appreciation + total_premium
    return_pct = (total_return / (initial_price * shares_owned)) * 100
    
    print(f"\n{'='*70}")
    print(f"6-MONTH COVERED CALL SIMULATION")
    print(f"{'='*70}")
    print(f"\nInitial Stock Price: ${initial_price:.2f}")
    print(f"Final Stock Price: ${current_price:.2f}")
    print(f"\nStock Appreciation: ${stock_appreciation:.2f}")
    print(f"Total Premium Collected: ${total_premium:.2f}")
    print(f"Total Return: ${total_return:.2f} ({return_pct:.2f}%)")
    print(f"\nAnnualized Return: {(return_pct * 2):.2f}%")  # Multiply by 2 for 6-month to annual
    print(f"\n{'='*70}\n")
    
    return df

# Run simulation
df_simulation = simulate_covered_call_strategy(50, months=6)

In [ ]:
# Display monthly results
print("\nMonthly Breakdown:")
print(df_simulation.to_string(index=False, float_format=lambda x: f'${x:.2f}' if isinstance(x, float) else x))

In [ ]:
# Visualize the simulation
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Stock Price and Strike Price Over Time
ax1.plot(df_simulation['Month'], df_simulation['End Price'], 
         marker='o', linewidth=2, markersize=8, color='#2E86AB', label='Stock Price')
ax1.plot(df_simulation['Month'], df_simulation['Strike'], 
         marker='s', linewidth=2, markersize=6, color='#E63946', 
         linestyle='--', label='Strike Price', alpha=0.7)

ax1.set_xlabel('Month', fontsize=11, fontweight='bold')
ax1.set_ylabel('Price ($)', fontsize=11, fontweight='bold')
ax1.set_title('Stock Price vs Strike Price Over Time', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Cumulative Premium Income
cumulative_premium = df_simulation['Premium'].cumsum()
ax2.bar(df_simulation['Month'], df_simulation['Premium'], 
        color='#06AED5', alpha=0.6, label='Monthly Premium')
ax2.plot(df_simulation['Month'], cumulative_premium, 
         marker='o', linewidth=2.5, markersize=8, color='#2B9348', 
         label='Cumulative Premium')

ax2.set_xlabel('Month', fontsize=11, fontweight='bold')
ax2.set_ylabel('Premium ($)', fontsize=11, fontweight='bold')
ax2.set_title('Premium Income Over Time', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Part 6: Key Takeaways and Best Practices

### Advantages of Covered Calls
1. **Generate Income**: Collect premium regardless of stock price movement
2. **Downside Protection**: Premium provides a cushion against small price drops
3. **Lower Risk**: Less risky than owning stock alone
4. **Flexible**: Can choose strike prices based on outlook

### Risks and Limitations
1. **Capped Upside**: Miss out on gains above strike price
2. **Stock Ownership Risk**: Still exposed to significant downside
3. **Opportunity Cost**: May be forced to sell at unfavorable time
4. **Tax Considerations**: May trigger short-term capital gains

### When to Use Covered Calls
- Neutral to slightly bullish on the stock
- Want to generate income from holdings
- Willing to sell stock at strike price
- Expect low to moderate volatility

### Strike Price Selection Guide
- **ITM Calls**: Higher premium, lower max profit, higher probability of assignment
- **ATM Calls**: Moderate premium and profit, 50% probability of assignment
- **OTM Calls**: Lower premium, higher max profit, lower probability of assignment

## Practice Exercise

Try modifying the parameters below to explore different scenarios:

In [ ]:
# Exercise: Create your own covered call scenario

# TODO: Modify these parameters
my_stock_price = 100  # Change this
my_strike_price = 105  # Change this
my_premium = 3  # Change this

# Generate payoff diagram for your scenario
stock_prices = np.linspace(my_stock_price * 0.7, my_stock_price * 1.3, 200)
pnl_values = [calculate_covered_call_pnl(price, my_stock_price, my_strike_price, my_premium) 
              for price in stock_prices]

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(stock_prices, pnl_values, linewidth=2.5, color='#2E86AB')
ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(x=my_stock_price, color='green', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(x=my_strike_price, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax.fill_between(stock_prices, pnl_values, 0, where=(np.array(pnl_values) > 0), alpha=0.3, color='green')
ax.fill_between(stock_prices, pnl_values, 0, where=(np.array(pnl_values) < 0), alpha=0.3, color='red')
ax.set_xlabel('Stock Price at Expiration ($)', fontsize=12, fontweight='bold')
ax.set_ylabel('Profit / Loss ($)', fontsize=12, fontweight='bold')
ax.set_title('Your Custom Covered Call Payoff Diagram', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

max_profit = (my_strike_price - my_stock_price) * 100 + my_premium * 100
breakeven = my_stock_price - my_premium
print(f"\nYour Scenario Analysis:")
print(f"Maximum Profit: ${max_profit:.2f}")
print(f"Maximum Profit %: {(max_profit / (my_stock_price * 100)) * 100:.2f}%")
print(f"Breakeven Price: ${breakeven:.2f}")
print(f"Downside Protection: {(my_premium / my_stock_price) * 100:.2f}%")

## Summary

In this tutorial, we've explored the covered call options strategy through:

1. Understanding the basic mechanics of the strategy
2. Calculating profit/loss scenarios mathematically
3. Visualizing payoff diagrams
4. Comparing different strike price selections
5. Simulating real-world multi-month strategies

The covered call is an excellent strategy for:
- Generating additional income from stock holdings
- Reducing portfolio volatility
- Outperforming in sideways or slightly bullish markets

Remember: This is a proof of concept for educational purposes. Real options trading involves additional factors like implied volatility, time decay, dividends, transaction costs, and tax implications.

### Further Learning
- Explore other option strategies (cash-secured puts, spreads, straddles)
- Study the Greeks (Delta, Theta, Vega, Gamma)
- Analyze real market data using options APIs
- Backtest strategies with historical data